# Options Order Book Simulation

Visualization of:
1. Brownian motion asset price
2. Poisson order book arrivals
3. Expiration times
4. Implied volatility forecast
5. Black-Scholes priced options

In [ ]:
import sys
sys.path.insert(0, '..')

from src import run_simulation, result_to_summary, result_to_price_path, orders_to_scatter, options_to_scatter
import matplotlib.pyplot as plt

In [ ]:
result = run_simulation(seed=42)
summary = result_to_summary(result)

## Summary

In [ ]:
print(f"Price: ${summary.initial_price:.2f} -> ${summary.final_price:.2f} ({summary.price_return:+.1f}%)")
print(f"Orders: {summary.n_underlying_orders} underlying, {summary.n_option_orders} options")
print(f"Options: {summary.n_calls} calls, {summary.n_puts} puts")
print(f"Vol: Market {summary.market_iv_pct:.1f}% | Trader {summary.trader_vol_pct:.1f}% | Spread {summary.vol_spread_pct:+.1f}%")

## 1. Price Path (GBM)

In [ ]:
path = result_to_price_path(result)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(path.times, path.prices, 'b-', lw=1)
ax.axhline(path.prices[0], color='gray', ls='--', lw=0.5)
ax.set_xlabel('Time (years)')
ax.set_ylabel('Price')
ax.set_title('Asset Price Path (GBM)')
plt.tight_layout()
plt.show()

## 2. Underlying Order Book

In [ ]:
scatter = orders_to_scatter(result.underlying_orders)

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(scatter.times, scatter.prices, s=scatter.sizes, c=scatter.colors, alpha=0.6)
ax.axhline(float(result.market.spot), color='blue', ls='--', lw=1, label=f'Spot: ${float(result.market.spot):.2f}')
ax.set_xlabel('Arrival Time')
ax.set_ylabel('Order Price')
ax.set_title('Order Book (green=bid, red=ask, size=quantity)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Expiration Times

In [ ]:
exp_days = result.expiration_times * 365

fig, ax = plt.subplots(figsize=(8, 2))
ax.barh(range(len(exp_days)), exp_days, color='steelblue')
ax.set_yticks(range(len(exp_days)))
ax.set_yticklabels([f'{int(d)}d' for d in exp_days])
ax.set_xlabel('Days to Expiry')
ax.set_title('Option Expirations')
plt.tight_layout()
plt.show()

## 4. Volatility Forecast

In [ ]:
vf = result.volatility_forecast

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(['Market IV', 'Trader'], [float(vf.market_iv)*100, float(vf.trader_forecast)*100], 
              color=['coral', 'steelblue'])
ax.set_ylabel('Volatility (%)')
ax.set_title(f'Volatility Forecast (spread: {vf.vol_spread*100:+.1f}%)')
for bar in bars:
    ax.annotate(f'{bar.get_height():.1f}%', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 5. Options Order Book

In [ ]:
opt_scatter = options_to_scatter(result.option_orders, float(result.market.spot))

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(opt_scatter.times, opt_scatter.prices, s=opt_scatter.sizes, c=opt_scatter.colors, alpha=0.6)
ax.axhline(float(result.market.spot), color='black', ls='--', lw=1, label='Spot')
ax.set_xlabel('Days to Expiry')
ax.set_ylabel('Strike')
ax.set_title('Options Order Book (blue=call, orange=put, size=BS price)')
ax.legend()
plt.tight_layout()
plt.show()

## Option Orders Table

In [ ]:
from src.typeAdapters import options_to_records

records = options_to_records(result.option_orders[:10])
print(f"{'Type':<6} {'Strike':>8} {'Expiry':>8} {'Price':>8} {'Side':<4}")
print('-' * 40)
for r in records:
    print(f"{r['option_type']:<6} {r['strike']:>8.2f} {r['expiry_days']:>6.0f}d {r['bs_price']:>8.2f} {r['side']:<4}")